# Verificación de Datos - ShopNow
---

### ¿Qué hace este cuaderno?

Están preparadas **cinco (05) pruebas automáticas** para revisar que los datos subidos están bien y no tienen fallos poco convenientes antes de empezar con los análisis. 

**¿Qué estamos vigilando?**
*   **Dinero:** Que no haya productos con precio 0 o descuentos que superen el precio (para no perder ganancias).
*   **Lógica:** Que no haya stock negativo y que las fechas de envío tengan sentido.
*   **Limpieza:** Que no se nos cuelen clientes duplicados por error.

---

### Cómo usarlo
1.  **Paso 1:** Conecta el JSON en la celda de conexión.
2.  **Paso 2:** Dale a "Ejecutar todo".
3.  **Resultados:** Si sale el mensaje de que no se han encontrado errores, es que todo está perfecto. Si sale el aviso  nos saldrá una tabla con los datos que será necesario corregir.

In [ ]:
# CELDA DE CONEXIÓN. 
import os
import pandas as pd
from google.cloud import bigquery
from dotenv import load_dotenv

load_dotenv()

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

project_id = os.getenv("GCP_PROJECT_ID")
dataset_id = os.getenv("BQ_DATASET_ID")

client = bigquery.Client(project=project_id)

print("Librerías cargadas y cliente de BigQuery listo.")

In [2]:
# Query de Verificación Nº1: ¿Hay clientes con el mismo email?

query_duplicados = """
SELECT 
    email, 
    COUNT(*) as total_repeticiones
FROM `tc-novarix-data.shopnow.customers`
GROUP BY email
HAVING total_repeticiones > 1
ORDER BY total_repeticiones DESC
"""

print("Buscando duplicados en la tabla de clientes...")

# Ejecución de la query
try:
    df_duplicados = client.query(query_duplicados).to_dataframe()

    if df_duplicados.empty:
        print("No se han encontrado emails duplicados.")
    else:
        print(f"Se han encontrado un total de {len(df_duplicados)} emails repetidos:")
        print(df_duplicados)
except NameError:
    print("Error: Primero tienes que ejecutar la celda de conexión con el JSON.")
except Exception as e:
    print(f"Ha ocurrido un error con la query: {e}")

# Nota de QA: Si no sale la tabla, es que el DE ha filtrado bien los datos sintéticos.

Buscando duplicados en la tabla de clientes...
Error: Primero tienes que ejecutar la celda de conexión con el JSON.


In [3]:
# Query de Verificación №2: ¿Existen productos con stock negativo?

query_stock_negativo = """
SELECT 
    product_id,
    product_name,
    stock
FROM `tc-novarix-data.shopnow.products`
WHERE stock < 0
ORDER BY stock ASC
"""

print("Buscando errores en el inventario...")

# Ejecución de la query
try:
    df_stock = client.query(query_stock_negativo).to_dataframe()

    if df_stock.empty:
        print("No se han encontrado productos con stock negativo.")
    else:
        print(f"Se han encontrado un total de {len(df_stock)} productos con stock negativo:")
        print(df_stock)

except NameError:
    print("Error: Primero tienes que ejecutar la celda de conexión con el JSON.")
except Exception as e:
    print(f"Ha ocurrido un error con la query: {e}")

# Nota de QA: Si no sale la tabla es que la lógica de inventario del DE es sólida.

Buscando errores en el inventario...
Error: Primero tienes que ejecutar la celda de conexión con el JSON.


In [4]:
# Query de Verificación №3: ¿Hay productos con precio igual a 0 o precios negativos?

query_precios_falsos = """
SELECT 
    product_id,
    product_name,
    price
FROM `tc-novarix-data.shopnow.products`
WHERE price <= 0
ORDER BY price ASC
"""

print("Buscando errores en los precios...")

# Ejecución de la query
try:
    df_precios = client.query(query_precios_falsos).to_dataframe()

    if df_precios.empty:
        print("Todos los productos tienen un precio correcto, superior a 0€.")
    else:
        print(f"Error importante. Se han detectado {len(df_precios)} productos con precio incorrecto:")
        print(df_precios)

except NameError:
    print("Error: Primero tienes que ejecutar la celda de conexión con el JSON.")
except Exception as e:
    print(f"Ha ocurrido un error con la query: {e}")

# Nota de QA: Esta prueba asegura que todos los productos tengan un precio 
# válido para evitar ventas a 0€ o errores en el catálogo.

Buscando errores en los precios...
Error: Primero tienes que ejecutar la celda de conexión con el JSON.


In [5]:
# Query de Verificación №4: ¿Hay descuentos mayores al precio del producto?

query_descuentos_altos = """
SELECT 
    order_id,
    product_id,
    price AS precio_original,
    discount AS descuento_aplicado,
    (price - discount) AS precio_final
FROM `tc-novarix-data.shopnow.orders`
WHERE discount > price
ORDER BY (price - discount) ASC
"""

print("Analizando precios finales...")

# Ejecución de la query
try:
    df_descuentos = client.query(query_descuentos_altos).to_dataframe()

    if df_descuentos.empty:
        print("No hay descuentos que superen el precio del artículo.")
    else:
        print(f"Se han detectado {len(df_descuentos)} pedidos con precio final negativo:")
        print(df_descuentos)

except NameError:
    print("Error: Primero tienes que ejecutar la celda de conexión con el JSON.")
except Exception as e:
    print(f"Ha ocurrido un error con la query: {e}")

# Nota de QA: Si esta tabla sale con datos, DE tiene que revisar los límites 
# de su generador de descuentos aleatorios.

Analizando precios finales...
Error: Primero tienes que ejecutar la celda de conexión con el JSON.


In [6]:
# Query de Verificación №5: Coherencia de estados y fechas de envío

query_coherencias = """
SELECT 
    order_id,
    status,
    created_at,
    delivered_at
FROM `tc-novarix-data.shopnow.orders`
WHERE 
    (status = 'Delivered' AND delivered_at IS NULL) OR (status = 'Pending' AND delivered_at IS NOT NULL)
ORDER BY created_at DESC
"""

print("Comprobando coherencias de logística...")

# Ejecución de la query
try:
    df_logistica = client.query(query_coherencias).to_dataframe()

    if df_logistica.empty:
        print("Todos los estados de envío coinciden con sus fechas.")
    else:
        print(f" ¡Incoherencia logística! Se han encontrado un total de {len(df_logistica)} pedidos con estados erróneos:")
        print(df_logistica)

except NameError:
    print("Error: Primero tienes que ejecutar la celda de conexión con el JSON.")
except Exception as e:
    print(f"Ha ocurrido un error con la query: {e}")

# Nota de QA: Esto detecta si el proceso de actualización de la base de datos 
# falló al marcar un pedido como entregado cuando no debería.

Comprobando coherencias de logística...
Error: Primero tienes que ejecutar la celda de conexión con el JSON.


## Fin de la Verificación
Si todos los bloques anteriores han mostrado el mensaje de éxito, los datos están listos para la fase de análisis y visualización.